# load the model

In [1]:
import torch
import nltk
nltk.download('punkt_tab')

# Define device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [2]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F
import re

# Mean Pooling with attention mask
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)


tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
model = AutoModel.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')





/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

In [3]:

from google.colab import files
import shutil, os, zipfile

model_id = "sentence-transformers/all-MiniLM-L6-v2"
save_dir = "/content/all-MiniLM-L6-v2"



# 导出到本地文件夹
os.makedirs(save_dir, exist_ok=True)
tokenizer.save_pretrained(save_dir)
model.save_pretrained(save_dir)

# 打包成 zip
zip_path = "/content/all-MiniLM-L6-v2.zip"
shutil.make_archive(zip_path.replace(".zip",""), "zip", save_dir)

# 下载到你的电脑
files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
tok = AutoTokenizer.from_pretrained(save_dir, local_files_only=True)
mdl = AutoModel.from_pretrained(save_dir, local_files_only=True)

In [5]:
model.eval()
model.to(device)
print(model)

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 384, padding_idx=0)
    (position_embeddings): Embedding(512, 384)
    (token_type_embeddings): Embedding(2, 384)
    (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-5): 6 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=384, out_features=384, bias=True)
            (key): Linear(in_features=384, out_features=384, bias=True)
            (value): Linear(in_features=384, out_features=384, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=384, out_features=384, bias=True)
            (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)


# Import terms and policy that need to be detect

In [6]:

test_paragraph = """


Apple Media Services Terms and Conditions

These terms and conditions create a contract between you and Apple (the “Agreement”). Please read the Agreement carefully.

TABLE OF CONTENTS

A. INTRODUCTION

B. PAYMENTS, TAXES, AND REFUNDS

C. ACCOUNT

D. PRIVACY

E. ACCESSIBILITY

F. SERVICES AND CONTENT USAGE RULES

G. TERMINATION AND SUSPENSION OF SERVICES

H. DOWNLOADS

I. SUBSCRIPTIONS

J. CONTENT AND SERVICE AVAILABILITY

K. THIRD-PARTY DEVICES AND EQUIPMENT

L. YOUR SUBMISSIONS TO OUR SERVICES

M. FAMILY SHARING

N. SEASON PASS AND MULTI-PASS

O. ADDITIONAL APP STORE TERMS

P. ADDITIONAL TERMS FOR APP STORE, APPLE BOOKS, APPLE PODCASTS, AND SELECT CONTENT

Q. ADDITIONAL APPLE MUSIC TERMS

R. ADDITIONAL APPLE FITNESS+ TERMS

S. CARRIER MEMBERSHIP

T. MISC. TERMS APPLICABLE TO ALL SERVICES

A. INTRODUCTION

This Agreement governs your use of Apple’s services (“Services” – e.g., and where available, App Store, Apple Arcade, Apple Books, Apple Fitness+, Apple Music, Apple News, Apple News+, Apple One, Apple Podcasts, Apple Podcasts Subscriptions, Apple Sports, Apple TV, Apple TV+, Apple TV Channels, Game Center, iTunes, and Shazam), through which you can buy, get, license, rent or subscribe to content, Apps (as defined below), and other in-app services (collectively, “Content”). Content may be offered through the Services by Apple or a third party. Our Services are available for your use in your country or territory of residence (“Home Country”). By creating an account for use of the Services in a particular country or territory you are specifying it as your Home Country. To use our Services, you need compatible hardware, software (latest version recommended and sometimes required) and Internet access (fees may apply). Our Services’ performance may be affected by these factors.

B. PAYMENTS, TAXES, AND REFUNDS

You can acquire Content on our Services for free or for a charge, either of which is referred to as a “Transaction.” Each Transaction is an electronic contract between you and Apple, and/or you and the entity providing the Content on our Services. However, if you are a customer of Apple Distribution International Ltd. or Apple Services Pte. Ltd., then that entity is the merchant of record for some Content you acquire from Apple Books, Apple Podcasts, or App Store as displayed on the product page and/or during the acquisition process for the relevant Service. In such case, you acquire the Content from Apple Distribution International Ltd. or Apple Services Pte. Ltd., as applicable, which is licensed by the Content provider (e.g., App Provider (as defined below), book publisher, etc.). When you make your first Transaction, we will ask you to choose how frequently we should ask for your password for future Transactions. On applicable Apple hardware, if you enable Touch ID for Transactions, we will ask you to authenticate all Transactions with your fingerprint, and if you enable Face ID for Transactions, we will ask you to authenticate all Transactions using facial recognition. Manage your password settings at any time by following these instructions: https://support.apple.com/HT204030.

Apple will charge your selected payment method for any paid Transactions, including any applicable taxes. If you have also added it to your Apple Wallet, Apple may charge your selected payment method in Apple Wallet using Apple Pay. You can associate multiple payment methods with your Apple Account, and you agree that Apple may store and charge those payment methods for Transactions. Your primary payment method appears at the top of your account settings payments page.

If your primary payment method cannot be charged for any reason (such as expiration or insufficient funds), you authorize Apple to attempt to charge your other eligible payment methods in order from top to bottom as they appear on your account settings payments page. If we cannot charge you, you remain responsible for any uncollected amounts, and we may attempt to charge you again or request that you provide another payment method. If you pre-order Content, you will be charged when the Content is delivered to you (unless you cancel prior to the Content’s availability). In accordance with local law, Apple may automatically update your payment information regarding your payment methods if provided such information by the payment networks or your financial institutions. Terms related to store credit and gift cards/codes are available here: https://www.apple.com/legal/internet-services/itunes/giftcards. For more details about how Transactions are billed, please visit http://support.apple.com/HT201359. All Transactions are final. Content prices may change at any time. If technical problems prevent or unreasonably delay delivery of Content, your exclusive and sole remedy is either replacement of the Content or refund of the price paid, as determined by Apple. From time to time, Apple may suspend or cancel payment or refuse a refund request if we find evidence of fraud, abuse, or unlawful or other manipulative behavior that entitles Apple to a corresponding counterclaim.

C. ACCOUNT

Using our Services and accessing your Content may require an Apple Account. An Apple Account (previously called an Apple ID) is the account you use across Apple’s ecosystem. Use of Game Center is subject to this Agreement and also requires a Game Center account. Your account is valuable, and you are responsible for maintaining its confidentiality and security. Apple is not responsible for any losses arising from the unauthorized use of your account. Please contact Apple if you suspect that your account has been compromised.

You must be age thirteen (13) (or equivalent minimum age in your Home Country, as set forth in the Apple Account creation process) to create an account and use our Services. Apple Accounts for persons under this age can be created by a parent or legal guardian using Family Sharing or by an approved educational institution, though certain devices may prevent such Apple Accounts from accessing certain Services on the device. A parent or legal guardian who is creating an account for a minor should review this Agreement with the minor to ensure that they both understand it.

You may add, notify, or remove a Legacy Contact for your Apple Account as described in http://support.apple.com/HT212360. A Legacy Contact’s access to your Apple Account is limited as described in http://support.apple.com/HT212361.

D. PRIVACY

Your use of our Services is subject to Apple’s Privacy Policy, which is available at https://www.apple.com/legal/privacy.

E. ACCESSIBILITY

To learn about accessibility features and other accessibility-related information related to the Services, visit https://www.apple.com/accessibility/labels.

F. SERVICES AND CONTENT USAGE RULES

Your use of the Services and Content must follow the rules set forth in this section (“Usage Rules”). Any other use of the Services and Content is a material breach of this Agreement. Apple may monitor your use of the Services and Content to ensure that you are following these Usage Rules.

All Services:

- You may use the Services and Content only for personal, noncommercial purposes (except as set forth in the App Store Content section below or as otherwise specified by Apple).

- Apple’s delivery of Services or Content does not transfer any commercial or promotional use rights to you, and does not constitute a grant or waiver of any rights of the copyright owners.

- You can use Content from up to five (5) different Apple Accounts on each device.

- For any Service, you can have up to ten (10) devices (but only a maximum of five (5) computers) signed in with your Apple Account at one time, though simultaneous streams or downloads of Content may be limited to a lower number of devices as set out below under Apple Music and Apple TV content. Each computer must also be authorized using the same Apple Account (to learn more about authorization of computers, visit https://support.apple.com/HT201251). Devices can be associated with a different Apple Account once every ninety (90) days.

- You may not manipulate play counts, downloads, ratings, or reviews via any means — such as (i) using a bot, script, or automated process; or (ii) providing or accepting any kind of compensation or incentive.

- You may not use any software, device, automated process, or any similar or equivalent manual process to scrape, copy, or perform measurement, analysis, or monitoring of, any portion of the Content or Services.

- It is your responsibility not to lose, destroy, or damage Content once downloaded. We encourage you to back up your Content regularly.

- You may not tamper with or circumvent any security technology included with the Services or Content.

- You may access our Services only using Apple’s software, and may not modify or use modified versions of such software.

- Video Content requires an HDCP connection.

Audio and Video Content Sales and Rentals:

- You can use Digital Rights Management (DRM)-free Content on a reasonable number of compatible devices that you own or control. DRM-protected Content can be used on up to five (5) computers and any number of devices that you sync to from those computers.

- Content rentals are viewable on a single device at a time, and must be played within thirty (30) days, and completed within forty-eight (48) hours of the start of play (stopping, pausing or restarting does not extend this period).

- You may burn an audio playlist of purchased music to disc for listening purposes up to seven (7) times; this limitation does not apply to DRM-free Content. Other Content may not be burned to disc.

- Purchased Content will generally remain available for you to download, redownload, or otherwise access from Apple. Though it is unlikely, subsequent to your purchase, Content may be removed from the Services and become unavailable for further download or access from Apple (for instance, because Apple loses its right from the Content provider to make it available). To ensure your ability to continue enjoying Content, we encourage you to download all purchased Content to a device in your possession and to back it up.

App Store Content:

- The term “Apps” includes apps and App Clips for any Apple platform and/or operating system, including any in-app purchases, extensions (such as keyboards), stickers, and subscriptions made available in such apps or App Clips.

- Individuals acting on behalf of a commercial enterprise, governmental organization or educational institution (an “Enterprise”) may download and sync non-Arcade Apps for use by either (i) a single individual on one (1) or more devices owned or controlled by an Enterprise; or (ii) multiple individuals on a single shared device owned or controlled by an Enterprise. For the sake of clarity, each device used serially or collectively by multiple users requires a separate license.

Apple Music:

- An Individual Apple Music membership allows you to stream on a single device at a time; a Family membership allows you or your Family members to stream on up to six (6) devices at a time.

Apple Arcade:

- Apple Arcade Apps may only be downloaded, or redownloaded, with a valid Apple Arcade trial or subscription.

- If your subscription ends, Apps downloaded via Apple Arcade will no longer be accessible to you.

Apple TV Content:

- For most channels, you can stream video Content on up to three (3) devices simultaneously.

- Learn more about Apple TV Content Usage Rules at https://support.apple.com/HT210074.

G. TERMINATION AND SUSPENSION OF SERVICES

If you fail, or Apple suspects that you have failed, to comply with any of the provisions of this Agreement, Apple may, without notice to you: (i) terminate this Agreement and/or your Apple Account, and you will remain liable for all amounts due under your Apple Account up to and including the date of termination; and/or (ii) terminate your license to the software; and/or (iii) preclude your access to the Services.

Apple further reserves the right to modify, suspend, or discontinue the Services (or any part or Content thereof) at any time with or without notice to you, and Apple will not be liable to you or to any third party should it exercise such rights.

H. DOWNLOADS

You may be limited in the amount of Content you may download, and some downloaded Content may expire after a given amount of time after downloaded or first played. Certain Content may not be available for download at all.

You may be able to redownload previously acquired Content (“Redownload”) to your devices that are signed in with the same Apple Account (“Associated Devices”). You can see Content types available for Redownload in your Home Country at https://support.apple.com/HT204632. Content may not be available for Redownload if that Content is no longer offered on our Services.

Content also may be removed from our Services at any time (for instance, because Apple loses its right from the Content provider to make it available), after which it cannot be downloaded, redownloaded, or otherwise accessed from Apple. We encourage you to back up your Content regularly.

I. SUBSCRIPTIONS

The Services and certain Apps may allow you to purchase access to Content or Services on a subscription basis (“Paid Subscriptions”). Paid Subscriptions automatically renew until cancelled in the Manage Subscriptions section of your account settings. To learn more about cancelling your subscriptions, visit https://support.apple.com/HT202039. You will be notified if the price of a Paid Subscription increases and, if required, your consent will be required to continue. You will be charged no more than twenty-four (24) hours prior to the start of the latest Paid Subscription period. If we cannot charge your payment method for any reason (such as expiration or insufficient funds), and you have not cancelled the Paid Subscription, you remain responsible for any uncollected amounts, and we will attempt to charge the payment method as you may update your payment method information. This may result in a change to the start of your next Paid Subscription period and may change the date on which you are billed for each period, as displayed on your receipt. We reserve the right to cancel your Paid Subscription if we are unable to successfully charge your payment method to renew your subscription. Certain Paid Subscriptions may offer a free trial prior to charging your payment method. If you decide to unsubscribe from a Paid Subscription before we start charging your payment method, cancel the subscription at least twenty-four (24) hours before the free trial ends.

If you start a free trial to a Paid Subscription offered by Apple as Content provider (an “Apple Paid Subscription”) and cancel before it ends, you cannot reactivate the free trial.

Free trials or free offers to Apple Paid Subscriptions, excluding iCloud, cannot be combined with any free trials or offers of Apple One. If you are in a free trial or free offer for any Apple Paid Subscriptions, and you subscribe to Apple One, your free trial(s) or offer(s) will not be paused even if you have access to such Apple Paid Subscription(s) through your Apple One subscription. You acknowledge that your free trial or free offer may expire while you are a Paid Subscriber to Apple One, and Apple shall have no obligation to reinstate, reimburse, or otherwise compensate you for any part of such expired free trial or free offer.

When your Paid Subscription to any Service or Content ends, you will lose access to any functionality or Content of that Service that requires a Paid Subscription.

J. CONTENT AND SERVICE AVAILABILITY

Terms found in this Agreement that relate to Services, Content types, features or functionality not available in your Home Country are not applicable to you unless and until they become available to you. To see the Content types available to you in your Home Country, go to the Services or visit https://support.apple.com/HT204411. Certain Services and Content available to you in your Home Country may not be available to you when traveling outside of your Home Country.

K. THIRD-PARTY DEVICES AND EQUIPMENT

You may not be able to use all features of the Services when accessing them on a non-Apple-branded device. Additionally, certain Services may require, direct, or suggest you use third-party equipment in some circumstances and/or for certain activities; such use is subject to the terms and conditions of such equipment and should be made in accordance with the applicable manufacturer’s instructions. By using the Services, you agree that Apple may automatically download and install minor updates to its software on third-party equipment from time to time.

L. YOUR SUBMISSIONS TO OUR SERVICES

Our Services may allow you to submit or post materials such as comments, ratings and reviews, pictures, videos, and podcasts (including associated metadata and artwork). Your use of such features must comply with the Submissions Guidelines below, which may be updated from time to time, and if we become aware of materials that violate our Submission Guidelines we will remove them. If you see materials that do not comply with the Submissions Guidelines, including any offensive, abusive, or illegal content, please let us know at reportaproblem.apple.com or by contacting Apple Support. Except to the extent prohibited by law, you hereby grant Apple a worldwide, royalty-free, perpetual, nonexclusive license to use the materials you submit within the Services and related marketing as well as to use the materials you submit for Apple internal purposes. Apple may monitor and decide to remove or edit any submitted material, including via automated content filters and/or human review.

Submissions Guidelines: You may not use the Services to:

- post any materials that (i) you do not have permission, right or license to use, or (ii) infringe on the rights of any third party;

- post objectionable, offensive, unlawful, deceptive, inaccurate, or harmful content;

- post personal, private or confidential information belonging to others;

- request personal information from a minor;

- impersonate or misrepresent your affiliation with another person, or entity;

- post or transmit spam, including but not limited to unsolicited or unauthorized advertising, promotional materials, or informational announcements;

- post, modify, or remove a rating or review in exchange for any kind of compensation or incentive;

- post a dishonest, abusive, harmful, misleading, or bad-faith rating or review, or a rating or review that is irrelevant to the Content being reviewed;

- plan or engage in any illegal, fraudulent, or manipulative activity.

M. FAMILY SHARING

The organizer of a Family (“Organizer”) must be eighteen (18) years (or the equivalent age of majority in their Home Country) or older and the parent or legal guardian of any Family member under age thirteen (13) (or the equivalent minimum age in their Home Country as set forth in the registration process). Apple devices are required for access to all of the Family Sharing features. Family Sharing allows eligible subscriptions to be shared among up to six (6) members of a Family. To learn more about Family Sharing visit: https://support.apple.com/HT201060.

Purchase Sharing: Family Sharing’s Purchase Sharing feature allows eligible Content to be shared among up to six (6) members of a Family. The Organizer invites other members to participate, and agrees to pay for all Transactions initiated by Family members. The Organizer’s eligible payment methods are used to pay for any Transaction initiated by a Family member (except when the Family member’s account has store credit, which is always used first). Family members are acting as agents for the Organizer when the Organizer’s eligible payment methods are used. The Organizer hereby agrees: (1) to pay for such Transactions; (2) that Transactions initiated by Family members are authorized; and (3) Transactions will be charged to eligible payment methods in the manner indicated in Section B above. Organizers are responsible for complying with their payment method contracts, and assume all risk related to sharing access to their eligible payment methods with Family members. A receipt or invoice for any Family member Transaction is sent to the initiating Family member and, if billed to the Organizer’s payment method, also to the Organizer.

Ask to Buy: Ask to Buy is a feature that allows an Organizer to approve Transactions initiated by a Family member under age eighteen (18) (or the equivalent age of majority in their Home Country). Content shared by Family members or acquired via content codes generally is not subject to Ask to Buy; content codes facilitating access to subscriptions are subject to Ask to Buy. The Organizer must be the parent or legal guardian of any Family member for whom Ask to Buy is activated. Ask to Buy is enabled by default for any Family member under the age of thirteen (13) (or the equivalent minimum age in their Home Country) and stays on until deactivated by the parent or legal guardian. If Ask to Buy is turned off after the Family member turns eighteen (18) years old (or reaches the age of majority in their Home Country), it cannot be turned on anymore. Ask to Buy does not require Purchase Sharing to be enabled.

Family Member changes: When a Family member leaves or is removed from the Family, the remaining Family members may no longer be able to access the former member’s Content, including Content acquired with the Organizer’s payment method.

Family Sharing Rules: You can only belong to one (1) Family at a time, and may join any Family no more than twice per year. You can change the Apple Account you associate with a Family no more than once every ninety (90) days. All Family members must share the same Home Country. Not all Content, including In-App Purchases, subscriptions, and some previously acquired Apps, are eligible for Purchase Sharing. Apple TV+, Apple TV Channels, Apple One Family, Apple One Premier, Apple Music Family, Apple Arcade, Apple News+, and Apple Fitness+ subscriptions are automatically enabled for Family Sharing. Subscriptions shared by a Family may be subject to Content usage limitations on a per subscription basis.

N. SEASON PASS AND MULTI-PASS

A Pass allows you to purchase and receive television Content as it becomes available. A Season Pass applies to television Content that has a limited number of episodes per season; a Multi-Pass applies to television Content that is available on an ongoing basis. The full price of a Season Pass or Multi-Pass is charged at the time of the Transaction. Season Pass or Multi-Pass Content is available for download up to ninety (90) days after the last episode becomes available. If a Content provider delivers to Apple fewer TV episodes than planned when you purchased a Season Pass, we will credit to your Apple Account the retail value of the corresponding number of episodes that were not provided to Apple.

O. ADDITIONAL APP STORE TERMS (EXCLUDING APPLE ARCADE APPS)

"""



# Start detecting

In [7]:
DISPLAY = {
    # the way of Tracking
    "tracking_on_site": "[ the way of Tracking ] On-site behaviour",
    "tracking_cross_site": "[ the way of Tracking ] Cross-site tracking",

    # type of data collected
    "type_name": "[ type of data collected ] Name",
    "type_email": "[ type of data collected ] Email",
    "type_phone": "[ type of data collected ] Phone",
    "type_ip": "[ type of data collected ] IP address",
    "type_device_id": "[ type of data collected ] Device ID",
    "type_cookies_adids": "[ type of data collected ] Cookies / ad IDs",
    "type_pages_viewed": "[ type of data collected ] Pages viewed",
    "type_clicks": "[ type of data collected ] Clicks",
    "type_app_events": "[ type of data collected ] App events",
    "type_crash_logs": "[ type of data collected ] Crash logs",
    "type_browser_os": "[ type of data collected ] Browser / OS",
    "type_screen_size": "[ type of data collected ] Screen size",
    "type_approx_loc_ip": "[ type of data collected ] Approximate location (IP-based)",
    "type_precise_loc_gps": "[ type of data collected ] Precise location (GPS)",
    "type_ugc": "[ type of data collected ] User-generated content (posts/photos/comments)",
    "type_order_details": "[ type of data collected ] Order details",
    "type_payment_tokens": "[ type of data collected ] Payment tokens",
    "type_sensitive": "[ type of data collected ] Sensitive data (health/biometrics/race/religion/sexual orientation)",

    # purpose of use
    "purpose_provide": "[ purpose of use ] Provide the service / fulfil a contract",
    "purpose_analytics": "[ purpose of use ] Analytics & product improvement",
    "purpose_marketing": "[ purpose of use ] Personalisation & marketing",
    "purpose_security": "[ purpose of use ] Security & fraud prevention",
    "purpose_legal": "[ purpose of use ] Legal compliance",

    # sharing with third parties
    "share_service_providers": "[ sharing with third parties ] Service providers (processors: cloud hosting / payments / support tools)",
    "share_adtech": "[ sharing with third parties ] Adtech  partners for targeted ads",
}

# --- ORDER for printing ---
field_order = [
    # tracking
    "tracking_on_site", "tracking_cross_site",
    # type of data collected
    "type_name", "type_email", "type_phone", "type_ip", "type_device_id",
    "type_cookies_adids", "type_pages_viewed", "type_clicks", "type_app_events",
    "type_crash_logs", "type_browser_os", "type_screen_size", "type_approx_loc_ip",
    "type_precise_loc_gps", "type_ugc", "type_order_details", "type_payment_tokens",
    "type_sensitive",
    # purposes
    "purpose_provide", "purpose_analytics", "purpose_marketing", "purpose_security", "purpose_legal",
    # sharing
    "share_service_providers", "share_adtech",
]

# --- SEMANTIC QUERY BANKS (only for non-"type" labels) ---
QUERIES = {
    # [ the way of Tracking ] On-site behaviour
    "tracking_on_site": [
        "We collect information about your activity in our services, such as views and interactions with content and ads.",
        "We automatically collect Personal Data about how you use the services we provide and your preferences.",
        "Google Analytics cookies measure how many website visitors we receive.",
        "We collect information about the interaction of your apps, browsers and devices with our services.",
        "We collect data on what songs you play in order to provide you with the content requested.",
    ],
    # [ the way of Tracking ] Cross-site tracking
    "tracking_cross_site": [
        "Google Analytics customers may enable linking your activity from one site with activity from other sites or apps.",
        "We collect activity on third-party sites and apps that use our services.",
        "You may be shown advertising on our site based on your browsing patterns on other sites.",
        "Additional Personal Data may be forwarded from third parties… to focus content we provide to you.",
        "You can control whether information about your activity on other sites and apps is saved and used.",
    ],

    # [ purpose of use ] Provide the service / fulfil a contract
    "purpose_provide": [
        "We use your information to deliver our services.",
        "To provide the Spotify Service. (legal basis: Performance of a Contract).",
        "Purchases: We use your Contact and Payment Information to process and fulfill purchases.",
        "Apple collects personal data necessary to power our services.",
        "We process the terms you search for in order to return results.",
    ],
    # [ purpose of use ] Analytics & product improvement
    "purpose_analytics": [
        "We use data for analytics and measurement to understand how our services are used.",
        "We may use personal data for auditing, data analysis, or troubleshooting.",
        "Analytics: we may automatically collect and use Analytical Information to continually improve your experience.",
        "Analyze and develop Stripe’s products and Services.",
        "Microsoft uses customer data to improve our products and services.",
    ],
    # [ purpose of use ] Personalisation & marketing
    "purpose_marketing": [
        "We use automated systems to provide customised search results and personalised ads.",
        "We may use your Personal Data to determine what products may be of interest and provide marketing communications.",
        "We use technology to automatically collect data that tells us how you use our websites and apps.",
        "We collect information to provide better services… like which ads you’ll find most useful.",
        "Tailored ads and interest-based advertising (controls available).",
    ],
    # [ purpose of use ] Security & fraud prevention
    "purpose_security": [
        "We use information to improve safety and reliability… including detecting, preventing and responding to fraud.",
        "We collect additional information… to identify bad actors and prevent fraudulent transactions.",
        "We and other organisations may share and use information to prevent fraud, money laundering.",
        "Use or disclose data… to detect, prevent or address fraud, security or technical issues.",
        "Apple uses personal data for security and fraud prevention.",
    ],
    # [ purpose of use ] Legal compliance
    "purpose_legal": [
        "We may share data to comply with any applicable law or regulation.",
        "Legal Obligation: processing is necessary for compliance with our legal obligations.",
        "Apple uses personal data… to comply with law.",
        "Comply with law, including anti-money-laundering and know-your-customer obligations.",
        "As required by law… comply with legal processes or requests from authorities.",
    ],

    # [ sharing with third parties ] Service providers
    "share_service_providers": [
        "We work with third-party service providers… for hosting, maintenance, backup, storage, payment processing, analysis and other services.",
        "We share your Personal Data with our service providers… to perform tasks on our behalf.",
        "We may share your Transaction Data with your bank or payment method provider… to process your transaction and prevent fraud.",
        "Third-party service providers receive Personal Data as necessary to perform their role.",
        "We disclose personal information to vendors and service providers to support our services.",
    ],
    # [ sharing with third parties ] Adtech partners for targeted ads
    "share_adtech": [
        "With your consent we share some information with our advertising partners.",
        "You may be shown advertising based on your browsing patterns on other sites.",
        "We use data… to help advertisers understand the performance of their ad campaigns.",
        "We may share aggregated data with third parties – like publishers, advertisers or connected websites.",
        "We don’t share information that personally identifies you with advertisers, unless you ask us to.",
    ],
}

# --- PER-FIELD THRESHOLDS for semantic banks ---
THRESH = {
    "tracking_on_site": 0.55,
    "tracking_cross_site": 0.55,
    "purpose_provide": 0.58,
    "purpose_analytics": 0.58,
    "purpose_marketing": 0.58,
    "purpose_security": 0.58,
    "purpose_legal": 0.58,
    "share_service_providers": 0.55,
    "share_adtech": 0.55,
}

# --- KEYWORD/REGEX ONLY (for all "type_*" fields) ---
# Helper: compile a simple OR pattern with word boundaries
def kw(*terms):
    # Escape non-regex terms and join; allow hyphen/space variants via character class where helpful
    escaped = []
    for t in terms:
        # tiny convenience: treat terms already containing regex (like IPv4/IPv6) as raw
        if any(ch in t for ch in r".*+?|\()[]{}^$\\"):
            escaped.append(t)
        else:
            escaped.append(re.escape(t))
    return r"(?:\b" + r"\b|\b".join(escaped) + r"\b)"

# IPv4 / IPv6 patterns
IPV4 = r"(?:\b(?:(?:25[0-5]|2[0-4]\d|1?\d?\d)\.){3}(?:25[0-5]|2[0-4]\d|1?\d?\d)\b)"
IPV6 = r"(?:\b(?:[A-F0-9]{1,4}:){2,7}[A-F0-9]{1,4}\b)"

REGEX = {
    "type_name": re.compile(kw("name", "first name", "last name", "full name"), re.I),
    "type_email": re.compile(kw("email", "email address"), re.I),
    "type_phone": re.compile(kw("phone", "telephone", "mobile number", "phone number"), re.I),

    "type_ip": re.compile(r"|".join([kw("ip address", "internet protocol address"), IPV4, IPV6]), re.I),

    "type_device_id": re.compile(
        kw("device id", "device identifier", "identifier for advertisers", "advertising id",
           "idfa", "gaid", "aaid", "android id", "imei", "mac address"), re.I),

    "type_cookies_adids": re.compile(
        kw("cookie", "cookies", "cookie id", "advertising id", "advertising identifier",
           "idfa", "gaid", "aaid"), re.I),

    "type_pages_viewed": re.compile(kw("pages viewed", "page views", "browsing history", "viewed pages"), re.I),

    "type_clicks": re.compile(kw("click", "clicks", "clickstream", "clicked"), re.I),

    "type_app_events": re.compile(kw("app events", "application events", "install events", "open events", "usage events"), re.I),

    "type_crash_logs": re.compile(kw("crash logs", "crash reports", "error logs", "diagnostic logs", "bug reports"), re.I),

    "type_browser_os": re.compile(kw("browser", "user agent", "operating system", "os", "platform"), re.I),

    "type_screen_size": re.compile(kw("screen size", "screen resolution", "display resolution", "viewport"), re.I),

    "type_approx_loc_ip": re.compile(
        kw("approximate location", "ip-based location", "location derived from ip", "country (based on ip)", "city (based on ip)"), re.I),

    "type_precise_loc_gps": re.compile(kw("precise location", "gps", "latitude", "longitude"), re.I),

    "type_ugc": re.compile(kw("user-generated content", "ugc", "posts", "photos", "comments", "reviews"), re.I),

    "type_order_details": re.compile(kw("order details", "order history", "purchase history", "items purchased"), re.I),

    "type_payment_tokens": re.compile(kw("payment token", "tokenized card", "payment method token", "nonce", "stripe token"), re.I),

    "type_sensitive": re.compile(
        kw("health", "biometric", "biometrics", "race", "religion", "sexual orientation", "precise location (gps)"), re.I),
}

In [8]:
sents = nltk.sent_tokenize(test_paragraph)

# --- RUN DETECTORS (semantic for QUERIES keys; regex-only for REGEX keys) ---
def bank_max_scores(query_list,sents):
    with torch.no_grad():
         enc = tokenizer(sents, padding=True, truncation=True, return_tensors="pt").to(device)
         out = model(**enc)
         sent_emb = mean_pooling(out, enc["attention_mask"])
         sent_emb = F.normalize(sent_emb, p=2, dim=1).to(device) # Move sent_emb to the device here
    with torch.no_grad():
        q_tok = tokenizer(query_list, padding=True, truncation=True, return_tensors="pt").to(device)
        q_out = model(**q_tok)
    q_emb = mean_pooling(q_out, q_tok["attention_mask"])
    q_emb = F.normalize(q_emb, p=2, dim=1)         # [Q, D]
    sims = sent_emb @ q_emb.T                      # [N, Q]
    max_scores, _ = sims.max(dim=1)                # [N]
    return max_scores.detach().cpu().tolist()

def regex_hits_for(field, sentences):
    pat = REGEX[field]
    return [bool(pat.search(s)) for s in sentences]

def compute_union_hits(field_hits, n_sents=None):
    """
    Sentence-wise OR across all fields.
    Returns: list[bool] of length n_sents.
    """
    if not field_hits:
        return [] if n_sents is None else [False] * n_sents

    if n_sents is None:
        n_sents = len(next(iter(field_hits.values())))

    union = [False] * n_sents
    for field, hits in field_hits.items():
        if len(hits) != n_sents:
            raise ValueError(f"Length mismatch for field '{field}': expected {n_sents}, got {len(hits)}")
        union = [u or h for u, h in zip(union, hits)]
    return union


def run_detectors(test_paragraph, field_order, QUERIES, THRESH,sents, REGEX):
    """
    Build per-field hit masks (semantic OR regex) and a union mask across fields.

    Args:
        sents: list[str] sentences to classify.
        field_order: iterable of field keys to evaluate (controls output order).
        QUERIES: dict[field -> list[str]] semantic query bank (optional per field).
        THRESH: dict[field -> float] similarity thresholds for fields present in QUERIES.
        REGEX: dict[field -> compiled_pattern] keyword/regex detector (optional per field).

    Returns:
        field_hits: dict[field -> list[bool]] length == len(sents)
        union_hits: list[bool] length == len(sents)
        detected_idx: list[int] indices where union_hits[i] is True
    """

    n = len(sents)
    field_hits = {}

    for field in field_order:
        # semantic route
        if field in QUERIES:
            scores = bank_max_scores(QUERIES[field],sents)  # uses your precomputed sent_emb
            # use field-specific threshold; raise if missing to avoid silent miscalibration
            if field not in THRESH:
                raise KeyError(f"Missing THRESH for semantic field '{field}'")
            sim_hits = [sc >= THRESH[field] for sc in scores]
        else:
            sim_hits = [False] * n

        # regex route
        if field in REGEX:
            rx_hits = regex_hits_for(field, sents)
        else:
            rx_hits = [False] * n

        # fuse
        field_hits[field] = [sh or rx for sh, rx in zip(sim_hits, rx_hits)]

    union_hits = compute_union_hits(field_hits, n_sents=n)
    detected_idx = [i for i, v in enumerate(union_hits) if v]
    return field_hits, union_hits, detected_idx

field_hits, union_hits, detected_idx = run_detectors(
    test_paragraph=test_paragraph, # Pass test_paragraph here
    field_order=field_order,
    QUERIES=QUERIES,
    THRESH=THRESH,
    sents=sents,
    REGEX=REGEX
)

# Results

In [10]:
import textwrap

def side_by_side_expected_with_fields(
    paragraph,
    sents,
    field_hits,
    union_hits,
    colw=70,
    field_order=("tracking_on_site", "tracking_cross_site", "type_name", "type_email", "type_phone",
                 "type_ip", "type_device_id", "type_cookies_adids", "type_pages_viewed", "type_clicks",
                 "type_app_events", "type_crash_logs", "type_browser_os", "type_screen_size",
                 "type_approx_loc_ip", "type_precise_loc_gps", "type_ugc", "type_order_details",
                 "type_payment_tokens", "type_sensitive", "purpose_provide", "purpose_analytics",
                 "purpose_marketing", "purpose_security", "purpose_legal", "share_service_providers",
                 "share_adtech"),
    DISPLAY=None,   # pass a dict to show friendly labels; defaults to raw keys
):
    assert len(sents) == len(union_hits), "sents and union_hits must be the same length"

    # left column: original paragraph, wrapped
    left_lines = textwrap.wrap(paragraph, width=colw)

    # right column: only detected sentences
    right_lines = []
    quote_w = colw - 8  # width for content inside "..."

    detected_idx = [i for i, hit in enumerate(union_hits) if hit]
    right_lines.append(f"Detected sentences [{len(detected_idx)}]:")

    for i in detected_idx:
        s = sents[i]
        # collect labels that fired for this sentence, keeping field_order
        raw_labels = [f for f in field_order if field_hits.get(f, []) and field_hits[f][i]]
        labels = [DISPLAY.get(f, f) for f in raw_labels] if isinstance(DISPLAY, dict) else raw_labels
        label_str = ", ".join(labels) if labels else "unlabeled"

        # wrap sentence but keep a single quoted block
        parts = []
        for line in s.splitlines() or [""]:
            wrapped = textwrap.wrap(line, width=quote_w) or [""]
            parts.extend(wrapped)
        if not parts:
            parts = [""]

        right_lines.append(f'    "{parts[0]}')
        for seg in parts[1:]:
            right_lines.append(f"      {seg}")
        right_lines[-1] += '"'
        right_lines.append(f"    [{label_str}]")
        right_lines.append("-" * colw)

    # pad both sides to same height and print side-by-side
    n = max(len(left_lines), len(right_lines))
    left_lines  += [""] * (n - len(left_lines))
    right_lines += [""] * (n - len(right_lines))
    for L, R in zip(left_lines, right_lines):
        print(L.ljust(colw), "|", R)


side_by_side_expected_with_fields(
    test_paragraph,
    sents,
    field_hits,
    union_hits,
    colw=70,
    field_order=field_order,   # your existing order list
    DISPLAY=DISPLAY            # optional pretty labels dict you defined earlier
)


   Apple Media Services Terms and Conditions  These terms and          | Detected sentences [12]:
conditions create a contract between you and Apple (the “Agreement”).  |     "To use our Services, you need compatible hardware, software
Please read the Agreement carefully.  TABLE OF CONTENTS  A.            |       (latest version recommended and sometimes required) and
INTRODUCTION  B. PAYMENTS, TAXES, AND REFUNDS  C. ACCOUNT  D. PRIVACY  |       Internet access (fees may apply)."
E. ACCESSIBILITY  F. SERVICES AND CONTENT USAGE RULES  G. TERMINATION  |     [unlabeled]
AND SUSPENSION OF SERVICES  H. DOWNLOADS  I. SUBSCRIPTIONS  J. CONTENT | ----------------------------------------------------------------------
AND SERVICE AVAILABILITY  K. THIRD-PARTY DEVICES AND EQUIPMENT  L.     |     "PAYMENTS, TAXES, AND REFUNDS
YOUR SUBMISSIONS TO OUR SERVICES  M. FAMILY SHARING  N. SEASON PASS    |       
AND MULTI-PASS  O. ADDITIONAL APP STORE TERMS  P. ADDITIONAL TERMS FOR |       You can acquire 

In [11]:
import csv
import os
import textwrap
from typing import Dict, List, Iterable, Optional, Tuple, Union

try:
    import pandas as pd  # optional, only used if return_df=True
except Exception:
    pd = None


def side_by_side_expected_with_fields(
    paragraph: str,
    sents: List[str],
    field_hits: Dict[str, List[bool]],
    union_hits: Iterable[bool],
    colw: int = 70,
    field_order: Tuple[str, ...] = (
        "tracking_on_site", "tracking_cross_site",
        "type_name", "type_email", "type_phone", "type_ip", "type_device_id",
        "type_cookies_adids", "type_pages_viewed", "type_clicks", "type_app_events",
        "type_crash_logs", "type_browser_os", "type_screen_size",
        "type_approx_loc_ip", "type_precise_loc_gps", "type_ugc",
        "type_order_details", "type_payment_tokens", "type_sensitive",
        "purpose_provide", "purpose_analytics", "purpose_marketing",
        "purpose_security", "purpose_legal",
        "share_service_providers", "share_adtech",
    ),
    DISPLAY: Optional[Dict[str, str]] = None,
    *,
    csv_path: Optional[str] = None,          # e.g. "/content/detections.csv"
    return_df: bool = False,                 # return a pandas.DataFrame (if pandas available)
    add_one_hot: bool = True,                # add one column per label (0/1)
) -> Union[None, "pd.DataFrame", List[dict]]:
    """
    Prints the side-by-side view. If `csv_path` is provided, writes a CSV of the
    right-hand detections. If `return_df` is True, returns a pandas DataFrame
    (or a list of dicts if pandas is unavailable).
    """
    union_hits = list(union_hits)
    assert len(sents) == len(union_hits), "sents and union_hits must be the same length"

    # --- Left column: original paragraph, wrapped
    left_lines = textwrap.wrap(paragraph, width=colw)

    # --- Right column: detected sentences + labels
    right_lines: List[str] = []
    quote_w = colw - 8  # width for content inside quotes
    detected_idx = [i for i, hit in enumerate(union_hits) if hit]
    right_lines.append(f"Detected sentences [{len(detected_idx)}]:")

    # Prepare CSV rows
    csv_rows: List[dict] = []
    base_columns = ["idx", "sentence", "labels_pretty", "labels_raw"]

    for i in detected_idx:
        s = sents[i]

        # Collect labels that fired for this sentence, in field_order
        fired_raw = [f for f in field_order if field_hits.get(f, []) and field_hits[f][i]]
        fired_pretty = [DISPLAY.get(f, f) for f in fired_raw] if isinstance(DISPLAY, dict) else fired_raw
        label_str = ", ".join(fired_pretty) if fired_pretty else "unlabeled"

        # --- Right-column pretty print
        parts = []
        for line in (s.splitlines() or [""]):
            wrapped = textwrap.wrap(line, width=quote_w) or [""]
            parts.extend(wrapped)
        if not parts:
            parts = [""]

        right_lines.append(f'    "{parts[0]}')
        for seg in parts[1:]:
            right_lines.append(f"      {seg}")
        right_lines[-1] += '"'
        right_lines.append(f"    [{label_str}]")
        right_lines.append("-" * colw)

        # --- CSV row
        row = {
            "idx": i,
            "sentence": s.replace("\n", " ").strip(),
            "labels_pretty": "|".join(fired_pretty),
            "labels_raw": "|".join(fired_raw),
        }
        if add_one_hot:
            for f in field_order:
                hit_list = field_hits.get(f, [])
                row[f] = int(bool(hit_list and len(hit_list) > i and hit_list[i]))
        csv_rows.append(row)

    # Pad and print side-by-side
    n = max(len(left_lines), len(right_lines))
    left_lines += [""] * (n - len(left_lines))
    right_lines += [""] * (n - len(right_lines))
    for L, R in zip(left_lines, right_lines):
        print(L.ljust(colw), "|", R)

    # Write CSV if requested
    if csv_path:
        os.makedirs(os.path.dirname(csv_path) or ".", exist_ok=True)
        # Ensure stable column order
        one_hot_cols = [f for f in field_order] if add_one_hot else []
        fieldnames = base_columns + one_hot_cols
        with open(csv_path, "w", newline="", encoding="utf-8-sig") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
            writer.writeheader()
            for r in csv_rows:
                writer.writerow(r)

    # Optional return
    if return_df:
        if pd is None:
            return csv_rows
        one_hot_cols = [f for f in field_order] if add_one_hot else []
        cols = base_columns + one_hot_cols
        # Guarantee all columns exist for DF construction
        for r in csv_rows:
            for c in cols:
                r.setdefault(c, 0 if c in field_order else "")
        return pd.DataFrame(csv_rows, columns=cols)

    return None

df = side_by_side_expected_with_fields(
    test_paragraph,
    sents,
    field_hits,
    union_hits,
    colw=70,
    field_order=field_order,
    DISPLAY=DISPLAY,
    csv_path="/content/detections.csv",  # writes the CSV
    return_df=True,                      # also get a DataFrame back
    add_one_hot=True                     # one column per label (0/1)
)

   Apple Media Services Terms and Conditions  These terms and          | Detected sentences [12]:
conditions create a contract between you and Apple (the “Agreement”).  |     "To use our Services, you need compatible hardware, software
Please read the Agreement carefully.  TABLE OF CONTENTS  A.            |       (latest version recommended and sometimes required) and
INTRODUCTION  B. PAYMENTS, TAXES, AND REFUNDS  C. ACCOUNT  D. PRIVACY  |       Internet access (fees may apply)."
E. ACCESSIBILITY  F. SERVICES AND CONTENT USAGE RULES  G. TERMINATION  |     [unlabeled]
AND SUSPENSION OF SERVICES  H. DOWNLOADS  I. SUBSCRIPTIONS  J. CONTENT | ----------------------------------------------------------------------
AND SERVICE AVAILABILITY  K. THIRD-PARTY DEVICES AND EQUIPMENT  L.     |     "PAYMENTS, TAXES, AND REFUNDS
YOUR SUBMISSIONS TO OUR SERVICES  M. FAMILY SHARING  N. SEASON PASS    |       
AND MULTI-PASS  O. ADDITIONAL APP STORE TERMS  P. ADDITIONAL TERMS FOR |       You can acquire 